# 03 launch args

Question: does the JVM host of Golem, the crate `golem-jvm`, read the arguments of a launch as the
java launcher of Temurin 17 reads them, and does a program run in its embedded JVM as it runs under
that launcher.

The java launcher of the Java home is the reference. A probe program prints, as JSON, its main
arguments, the options the JVM records in `getInputArguments`, the class path, the module path,
`sun.java.command`, `sun.java.launcher`, the name of its thread, whether the options of the module
system took effect and the modules of the boot layer. For argument files with quotes, escapes,
comments, continued lines and @ references, for the launcher options of the module system and the
class path, and for the real `user_jvm_args.txt` and `unix_args.txt` of the installed Forge server,
the notebook compares (a) what `golem-devtool argfile` and `launch-spec` read with what the
launcher handed to the JVM, and (b) the report of the probe under `golem-devtool jvm-run`, inside
the embedded JVM, with its report under java. It also checks the options Golem refuses,
`System.exit`, an exception of main, a thread that outlives main, the shutdown hooks, a fatal error
and the errors of the program `golem`.

Golem differs from the launcher on purpose in five points, and the cells that allow a difference
name it: the hook options `exit` and `abort` appear in `getInputArguments`, Golem drops
`--disable-@files` where the JVM of the launcher refuses it, Golem refuses a NUL byte in an argument
file, and the variables `JDK_JAVA_OPTIONS` and `CLASSPATH` of the launcher do not apply. Golem also
leaves out the warnings that the launcher prints for four old options, which change nothing else.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The Java home whose launcher is the reference, relative to the root.
java_home = "tests/.cache/jdk-17.0.21+9"
# The installed Forge server whose argument files the notebook reads and never writes.
forge_server = "tests/.cache/forge-server"
# The version of the Java home and the Forge version of the server.
java_version = "17.0.21"
forge_version = "1.20.1-47.4.18"

## Setup

The cell finds the repository, builds `golem-devtool` and `golem`, checks the Java home and the
Forge server, and prints the versions. Every process runs without the variables that change a launch
from outside its arguments, `JAVA_TOOL_OPTIONS`, `_JAVA_OPTIONS`, `JDK_JAVA_OPTIONS` and
`CLASSPATH`, so both sides see the arguments alone, and a case that checks one sets it.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import QUIET, paths, run

P = paths(golem_root)
JAVA_HOME = (P.root / java_home).resolve()
FORGE = (P.root / forge_server).resolve()
JAVA = JAVA_HOME / "bin" / "java"
UNIX_ARGS = "libraries/net/minecraftforge/forge/%s/unix_args.txt" % forge_version
assert JAVA.is_file(), "no Java home at %s: install Temurin %s there" % (java_home, java_version)
assert (FORGE / UNIX_ARGS).is_file(), "no Forge server %s at %s" % (forge_version, forge_server)

LAUNCH_VARIABLES = ["JAVA_TOOL_OPTIONS", "_JAVA_OPTIONS", "JDK_JAVA_OPTIONS", "CLASSPATH", "GOLEM_HOST_JAR"]


def launch(argv, cwd, variables=None, unset=(), timeout=180):
    # Runs argv with the quiet environment and without the variables of a launch.
    environment = {k: v for k, v in os.environ.items() if k not in LAUNCH_VARIABLES and k not in unset}
    environment.update(QUIET)
    environment.update(variables or {})
    return subprocess.run([str(part) for part in argv], cwd=cwd, env=environment,
                          capture_output=True, text=True, timeout=timeout)


build = run(["cargo", "build", "--quiet", "-p", "golem-devtool", "-p", "golem"], cwd=P.root, timeout=1200)
assert build.returncode == 0, "cargo build failed:\n" + build.stderr
# The target directory of cargo, which CARGO_TARGET_DIR or build.target-dir may move.
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
GOLEM = Path(metadata["target_directory"]) / "debug" / "golem"
assert DEVTOOL.is_file() and GOLEM.is_file(), "cargo built no golem-devtool or golem in %s" % metadata["target_directory"]
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-03-"))

version = launch([JAVA, "-version"], SCRATCH)
assert ('"%s"' % java_version) in version.stderr, "the Java home is not %s:\n%s" % (java_version, version.stderr)
print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("java:", version.stderr.strip().splitlines()[1])
print("java home:", java_home)
print("forge server:", forge_server, forge_version)

## The probe

The probe is a Java program that the cell writes into the temporary directory and compiles with
the `javac` of the Java home: `Probe.class` at the top of the directory, so the class path `.` finds
it, and `probe.jar`, which is also a Java agent whose `premain` prints the report and halts the JVM
before the main class of Forge runs. Main reads its arguments as actions: `exit=N`, `throw`,
`thread`, `hook` and `oom`. A second jar, `golem-host.jar`, holds a class
`com.skytekx.golem.host.Host` and names the automatic module `com.skytekx.golem.host`, as the host
jar of the bridge does.

In [ ]:
PROBE = r"""
import java.lang.instrument.Instrumentation;
import java.lang.management.ManagementFactory;
import java.util.ArrayList;
import java.util.List;
import java.util.Map;
import java.util.Optional;
import java.util.TreeMap;

public final class Probe {
	public static void main(String[] args) throws Exception {
		System.out.println(report("main", args));
		for (String arg : args) {
			act(arg);
		}
	}

	public static void premain(String options, Instrumentation instrumentation) {
		System.out.println(report("agent", new String[0]));
		System.out.flush();
		Runtime.getRuntime().halt(0);
	}

	private static void act(String arg) throws Exception {
		if (arg.startsWith("exit=")) {
			System.exit(Integer.parseInt(arg.substring(5)));
		} else if (arg.equals("throw")) {
			throw new IllegalStateException("the probe throws");
		} else if (arg.equals("thread")) {
			new Thread(() -> {
				pause(300);
				System.out.println("probe: the thread ends");
			}, "probe-thread").start();
		} else if (arg.equals("hook")) {
			Runtime.getRuntime().addShutdownHook(new Thread(() -> System.out.println("probe: the shutdown hook runs")));
		} else if (arg.equals("segv")) {
			java.lang.reflect.Field field = sun.misc.Unsafe.class.getDeclaredField("theUnsafe");
			field.setAccessible(true);
			((sun.misc.Unsafe) field.get(null)).putAddress(0L, 0L);
		} else if (arg.equals("oom")) {
			List<long[]> hold = new ArrayList<>();
			while (true) {
				hold.add(new long[1 << 20]);
			}
		}
	}

	private static void pause(long millis) {
		try {
			Thread.sleep(millis);
		} catch (InterruptedException e) {
			Thread.currentThread().interrupt();
		}
	}

	private static String report(String where, String[] args) {
		Map<String, Object> out = new TreeMap<>();
		List<String> input = ManagementFactory.getRuntimeMXBean().getInputArguments();
		out.put("args", List.of(args));
		out.put("probe", where);
		out.put("input", input);
		out.put("class_path", System.getProperty("java.class.path"));
		out.put("module_path", System.getProperty("jdk.module.path"));
		out.put("command", System.getProperty("sun.java.command"));
		out.put("launcher", System.getProperty("sun.java.launcher"));
		out.put("thread", Thread.currentThread().getName());
		out.put("opens_java_lang", Object.class.getModule().isOpen("java.lang", Probe.class.getModule()));
		out.put("checks", checks(input));
		List<Object> modules = new ArrayList<>();
		ModuleLayer.boot().modules().stream()
				.filter(m -> !m.getName().startsWith("java.") && !m.getName().startsWith("jdk."))
				.sorted((a, b) -> a.getName().compareTo(b.getName()))
				.forEach(m -> {
					Map<String, Object> module = new TreeMap<>();
					module.put("name", m.getName());
					module.put("automatic", m.getDescriptor().isAutomatic());
					module.put("exported", m.getPackages().stream().filter(m::isExported).sorted().toList());
					modules.add(module);
				});
		out.put("boot_modules", modules);
		return json(out);
	}

	// Whether every --add-opens and --add-exports that the JVM recorded took effect.
	private static List<Object> checks(List<String> input) {
		List<Object> checks = new ArrayList<>();
		for (String option : input) {
			String kind = option.startsWith("--add-opens=") ? "--add-opens="
					: option.startsWith("--add-exports=") ? "--add-exports=" : null;
			if (kind == null) {
				continue;
			}
			String value = option.substring(kind.length());
			int slash = value.indexOf('/');
			int equals = value.indexOf('=');
			if (slash < 0 || equals < slash) {
				continue;
			}
			Optional<Module> source = ModuleLayer.boot().findModule(value.substring(0, slash));
			String pkg = value.substring(slash + 1, equals);
			for (String target : value.substring(equals + 1).split(",")) {
				Module to = target.equals("ALL-UNNAMED") ? Probe.class.getModule()
						: ModuleLayer.boot().findModule(target).orElse(null);
				boolean holds = source.isPresent() && to != null && (kind.equals("--add-opens=")
						? source.get().isOpen(pkg, to) : source.get().isExported(pkg, to));
				checks.add(List.of(kind + value.substring(0, equals + 1) + target, holds));
			}
		}
		return checks;
	}

	private static String json(Object value) {
		if (value == null) {
			return "null";
		}
		if (value instanceof Boolean || value instanceof Number) {
			return value.toString();
		}
		if (value instanceof Map<?, ?> map) {
			StringBuilder out = new StringBuilder("{");
			for (Map.Entry<?, ?> entry : map.entrySet()) {
				out.append(out.length() > 1 ? ", " : "").append(json(entry.getKey().toString()));
				out.append(": ").append(json(entry.getValue()));
			}
			return out.append('}').toString();
		}
		if (value instanceof List<?> list) {
			StringBuilder out = new StringBuilder("[");
			for (Object item : list) {
				out.append(out.length() > 1 ? ", " : "").append(json(item));
			}
			return out.append(']').toString();
		}
		StringBuilder out = new StringBuilder("\"");
		for (char c : value.toString().toCharArray()) {
			if (c == '"' || c == '\\') {
				out.append('\\').append(c);
			} else if (c < 0x20) {
				out.append(String.format("\\u%04x", (int) c));
			} else {
				out.append(c);
			}
		}
		return out.append('"').toString();
	}
}
"""

HOST = r"""
package com.skytekx.golem.host;

public final class Host {
	private Host() {
	}
}
"""

JAVAC = JAVA_HOME / "bin" / "javac"
JAR = JAVA_HOME / "bin" / "jar"
(SCRATCH / "Probe.java").write_text(PROBE.lstrip(), encoding="utf-8")
(SCRATCH / "host" / "com" / "skytekx" / "golem" / "host").mkdir(parents=True)
(SCRATCH / "host" / "com" / "skytekx" / "golem" / "host" / "Host.java").write_text(HOST.lstrip(), encoding="utf-8")
(SCRATCH / "probe.mf").write_text("Premain-Class: Probe\n")
(SCRATCH / "host.mf").write_text("Automatic-Module-Name: com.skytekx.golem.host\n")
for step in [
    [JAVAC, "-d", SCRATCH, SCRATCH / "Probe.java"],
    [JAR, "--create", "--file", SCRATCH / "probe.jar", "--manifest", SCRATCH / "probe.mf", "-C", SCRATCH, "Probe.class"],
    [JAVAC, "-d", SCRATCH / "host", SCRATCH / "host" / "com" / "skytekx" / "golem" / "host" / "Host.java"],
    [JAR, "--create", "--file", SCRATCH / "golem-host.jar", "--manifest", SCRATCH / "host.mf", "-C", SCRATCH / "host", "com/skytekx/golem/host/Host.class"],
]:
    done = launch(step, SCRATCH)
    assert done.returncode == 0, "%s failed:\n%s%s" % (Path(step[0]).name, done.stdout, done.stderr)
shutil.rmtree(SCRATCH / "host")
print("probe:", sorted(path.name for path in SCRATCH.iterdir()))

## The helpers

`java` runs the launcher in a directory, `embedded` runs the same arguments through
`golem-devtool jvm-run` with that directory as the base, and `spec` asks `golem-devtool launch-spec`
for the launch. HotSpot records every option of `JNI_CreateJavaVM` in `getInputArguments` but
`-Djava.class.path`, `-Dsun.java.command` and `-Dsun.java.launcher` and the properties that start
with them (`parse_each_vm_init_arg` in `arguments.cpp`), so `recorded` drops those from a launch.
The hooks of the host are options, `exit` and `abort`, so HotSpot records them behind the options of
the launch, and `HOOKS` is what the embedded JVM adds there.

In [ ]:
HOOKS = ["exit", "abort"]
UNRECORDED = ("-Djava.class.path", "-Dsun.java.command", "-Dsun.java.launcher")


def java(arguments, cwd=None, variables=None):
    return launch([JAVA, *arguments], cwd or SCRATCH, variables)


def embedded(arguments, cwd=None, variables=None, jars=()):
    extra = [part for jar in jars for part in ("--add-module-path", jar)]
    return launch([DEVTOOL, "jvm-run", "--java-home", JAVA_HOME, "--base", cwd or SCRATCH, *extra, "--", *arguments], P.root, variables)


def spec(arguments, cwd=None):
    done = launch([DEVTOOL, "launch-spec", "--base", cwd or SCRATCH, "--", *arguments], P.root)
    return (json.loads(done.stdout) if done.returncode == 0 else None), done


def reports(done):
    # The reports of the probe in the standard output of a run, in their order.
    return [json.loads(line) for line in done.stdout.splitlines() if line.startswith('{"args"')]


def outcome(done):
    # The outcome line of jvm-run, which it prints after the JVM ended.
    lines = [json.loads(line) for line in done.stdout.splitlines() if line.startswith('{"destroy_code"')]
    return lines[-1] if lines else None


def recorded(options):
    return [option for option in options if not option.startswith(UNRECORDED)]


def last(options, prefix):
    values = [option[len(prefix):] for option in options if option.startswith(prefix)]
    return values[-1] if values else None


def write(files, base=None):
    for name, text in files.items():
        path = (base or SCRATCH) / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(text if isinstance(text, bytes) else text.encode("utf-8"))


def without_hooks(report):
    # A report of the embedded JVM with the hook options taken out, which is what the launcher
    # reports, so that every other field must be equal.
    assert report["input"][-2:] == HOOKS, "the hooks are not the last recorded options: %s" % report["input"][-4:]
    return dict(report, input=report["input"][:-2])


def show(rows, header):
    width = max(len(row[0]) for row in rows)
    print("%-*s  %s" % (width, header[0], "  ".join(header[1:])))
    for row in rows:
        print("%-*s  %s" % (width, row[0], "  ".join(str(cell) for cell in row[1:])))

## Argument files

Each case is the text of an argument file whose first token is the main class `Probe`, so the
launcher hands every other token to main as an argument, unchanged by any option rule. The tokens of
`golem-devtool argfile` must be `Probe` and the main arguments of the probe under java, and under
the embedded JVM. The launcher reads a file in blocks of 4096 bytes, and the case across the first
block holds the token that a `#` splits at the edge of the block, a quirk of `args.c` that Golem
keeps.

In [ ]:
ARGFILES = [
    ("white space", "Probe a b\tc\fd\ne\r\nf  \n"),
    ("double quotes", 'Probe "a b" "c\'d"\n'),
    ("single quotes", "Probe 'a \"b\" c'\n"),
    ("quotes in the middle of a token", "Probe pre\"fix mid\"post x'y'z\n"),
    ("escapes in quotes", 'Probe "a\\nb\\tc\\rd\\fe\\\\f\\"g\\q"\n'),
    ("a backslash outside quotes", "Probe c:\\path\\file \\\"x\n"),
    ("a continued line", 'Probe "one \\\n    two" three\n'),
    ("a continued line with CR LF", 'Probe "one \\\r\n\t  two"\r\n'),
    ("comments", "Probe a # a comment\nb #c\n# a line of comment\nc"),
    ("a hash in quotes", "Probe \"a#b\" '#c'\n"),
    ("a hash inside a token", "Probe abc#def\nnext\n"),
    ("a quoted part before a hash", "Probe 'x'abc#c\nnext\n"),
    ("a quote the line leaves open", 'Probe "open quote \n  next "unterminated'),
    ("empty quotes", "Probe \"\" '' x\"\"y\n"),
    ("empty quotes at the end of the file", 'Probe a ""'),
    ("an @ inside an argument file", "Probe @other.txt @@x @\n"),
    ("a token across the first block", "Probe " + "x" * 4083 + " abcdefghij#comment\nnext\n"),
    ("UTF-8", "Probe grüße 'ünïcode quoted' 日本\n"),
    ("no tokens", "Probe\n# only a comment\n\n"),
]

rows, failures = [], []
for index, (name, text) in enumerate(ARGFILES):
    file = "argfile-%02d.txt" % index
    write({file: text})
    tokens = launch([DEVTOOL, "argfile", SCRATCH / file], P.root)
    golem = json.loads(tokens.stdout)["tokens"] if tokens.returncode == 0 else None
    launcher = reports(java(["@" + file]))
    inside = reports(embedded(["@" + file]))
    expected = ["Probe"] + launcher[0]["args"] if launcher else None
    ok = expected is not None and golem == expected and inside and inside[0]["args"] == launcher[0]["args"]
    rows.append((name, len(golem or []), "ok" if ok else "WRONG"))
    if not ok:
        failures.append((name, expected, golem, inside[0]["args"] if inside else tokens.stderr))
show(rows, ("case", "tokens", ""))
for failure in failures:
    print("\n--- %s\nlauncher: %r\nargfile:  %r\nembedded: %r" % failure)
assert not failures, "%d of %d argument files read differently" % (len(failures), len(ARGFILES))

### A NUL byte

The launcher reads the tokens as C strings, so it cuts a token at a NUL byte and runs. Golem refuses
the file, since no option string of `JNI_CreateJavaVM` and no Java argument built from it can carry
the byte, and a cut token is never what the file means. This is one of the allowed differences.

In [ ]:
write({"nul.txt": b"Probe a\x00b c\n"})
launcher = reports(java(["@nul.txt"]))
assert launcher and launcher[0]["args"] == ["a", "c"], "the launcher no longer cuts at NUL: %s" % launcher
refused = launch([DEVTOOL, "argfile", SCRATCH / "nul.txt"], P.root)
assert refused.returncode == 1 and "NUL byte" in refused.stderr, refused.stdout + refused.stderr
inside = embedded(["@nul.txt"])
assert inside.returncode == 1 and "NUL byte" in inside.stderr, inside.stdout + inside.stderr
print("launcher: cuts the token at the NUL byte, golem:", refused.stderr.strip().rsplit(": ", 1)[-1])

## Launches

Each case is a command line of the launcher, with the files it needs. The cell checks (a) the launch
of `launch-spec` against the report of the probe under java: the recorded options in their order,
the main arguments, `sun.java.command`, the class path, the module path and `sun.java.launcher`, and
(b) the report of the probe in the embedded JVM against the one under java, which must be equal but
for the hooks. A case may set a variable that both sides read: `JAVA_TOOL_OPTIONS` is read by the
JVM itself, so it applies inside `JNI_CreateJavaVM` as it does under the launcher.

In [ ]:
JAR_BYTES = (SCRATCH / "probe.jar").read_bytes()
LAUNCHES = [
    ("options and arguments", ["-Xmx64m", "-Dprobe.a=1", "-cp", ".", "Probe", "a", "b c", ""], {}, {}),
    ("module options after a space", ["--add-opens", "java.base/java.lang=ALL-UNNAMED", "--add-exports", "java.base/sun.security.util=ALL-UNNAMED",
                                      "--add-reads", "java.base=ALL-UNNAMED", "--add-modules", "java.sql", "--enable-native-access", "ALL-UNNAMED", "Probe"], {}, {}),
    ("module options with =", ["--add-opens=java.base/java.util=ALL-UNNAMED,java.sql", "--add-exports=java.base/sun.nio.ch=ALL-UNNAMED", "--add-modules=java.sql", "Probe"], {}, {}),
    ("module paths", ["-p", "mods", "--module-path=mods2", "--upgrade-module-path", "upgrade", "--patch-module", "java.base=patch",
                      "--limit-modules", "java.base,java.management,java.instrument", "Probe"], {}, {}),
    ("class path options", ["-classpath", "none", "--class-path", "none", "--class-path=.", "Probe"], {}, {}),
    ("a class path of wildcards", ["-cp", ".:lib/*:missing/*:*", "Probe"], {"lib/a.jar": JAR_BYTES, "lib/B.JAR": JAR_BYTES, "lib/c.Jar": JAR_BYTES, "lib/d.zip": JAR_BYTES}, {}),
    ("the class path property", ["-Djava.class.path=.", "Probe"], {}, {}),
    ("no class path", ["Probe"], {}, {}),
    ("-client and -server", ["-client", "-server", "-Dprobe.a=1", "Probe", "-server"], {}, {}),
    ("-client and -server behind -J", ["-J-server", "-J-client", "-Dprobe.a=1", "Probe"], {}, {}),
    ("old forms of options", ["-verbosegc", "-noclassgc", "-ms64m", "-mx128m", "-ss2m", "-checksource", "-cs", "-noasyncgc",
                              "-verify", "-verifyremote", "-noverify", "-Xfuture", "-debug", "-Xdiag", "--show-module-resolution", "Probe"], {}, {}),
    ("-Xss and its old form", ["-Xss16m", "-ss4m", "Probe"], {}, {}),
    ("options that pass as they are", ["--illegal-access=permit", "--enable-preview", "-ea", "-Xshare:auto", "-XX:+UseSerialGC", "-Xlog:disable", "Probe"], {}, {}),
    ("@files before the main class", ["@options.txt", "Probe", "@options.txt", "@@escaped", "@"], {"options.txt": "-Dprobe.file=1 -cp .\n"}, {}),
    ("the main class in an @file", ["@main.txt", "@options.txt"], {"main.txt": "-cp . Probe first\n", "options.txt": "-Dprobe.file=1\n"}, {}),
    ("an escaped @ in a value", ["-cp", "@@.:.", "-Dprobe.a=1", "Probe"], {}, {}),
    ("the properties of the launcher", ["-Dsun.java.command=other", "-Dsun.java.launcher=other", "Probe", "x"], {}, {}),
    ("JAVA_TOOL_OPTIONS", ["-Dprobe.a=1", "Probe"], {}, {"JAVA_TOOL_OPTIONS": "-Dprobe.tool=1 -Xmx96m"}),
]

rows, failures = [], []
for name, arguments, files, variables in LAUNCHES:
    write(files)
    golem, asked = spec(arguments)
    launcher = reports(java(arguments, variables=variables))
    inside = embedded(arguments, variables=variables)
    problems = []
    if golem is None:
        problems.append("launch-spec failed: " + asked.stderr)
    if len(launcher) != 1:
        problems.append("the launcher did not run the probe")
    if not problems:
        report = launcher[0]
        tool = variables.get("JAVA_TOOL_OPTIONS", "").split()
        checks = [
            ("recorded options", tool + recorded(golem["options"]), report["input"]),
            ("main arguments", golem["arguments"], report["args"]),
            ("sun.java.command", " ".join([golem["main_class"], *golem["arguments"]]), report["command"]),
            ("class path", last(golem["options"], "-Djava.class.path="), report["class_path"]),
            ("module path", last(golem["options"], "--module-path="), report["module_path"]),
            ("sun.java.launcher", "SUN_STANDARD", report["launcher"]),
            ("main thread", "main", report["thread"]),
            ("checks hold", True, all(holds for _, holds in report["checks"])),
        ]
        problems += ["%s: golem %r, launcher %r" % check for check in checks if check[1] != check[2]]
        embedded_reports = reports(inside)
        if len(embedded_reports) != 1 or without_hooks(embedded_reports[0]) != report:
            problems.append("embedded report differs:\n  %s\n  %s" % (embedded_reports, report))
        if outcome(inside) != {"destroy_code": 0, "exception": None, "main": "returned"}:
            problems.append("embedded outcome %s" % outcome(inside))
    rows.append((name, len(golem["options"]) if golem else "-", "ok" if not problems else "WRONG"))
    if problems:
        failures.append((name, problems))
show(rows, ("case", "options", ""))
for name, problems in failures:
    print("\n--- %s\n%s" % (name, "\n".join(problems)))
assert not failures, "%d of %d launches differ" % (len(failures), len(LAUNCHES))

## Launches that fail

A launch that the launcher refuses or that the JVM refuses must fail the same way in Golem: with
the same exit status and the message of the launcher or the JVM. An option without its value and an
@file that does not open are errors of the launch itself, which `launch-spec` reports. A main class
that does not load, or an option that the JVM does not know, fails inside the JVM, under both. An @
reference inside an argument file is no reference, so its token is the main class, under both.

In [ ]:
FAILURES = [
    ("a class path without its value", ["-cp", "-Dx=1", "Probe"], {}, "-cp requires class path specification"),
    ("a module path without its value", ["-p"], {}, "-p requires module path specification"),
    ("a module option without its value", ["--add-opens"], {}, "--add-opens requires modules to be specified"),
    ("a module option with = and no value", ["--add-modules=", "Probe"], {}, "--add-modules= requires modules to be specified"),
    ("an @file that does not exist", ["@missing.txt", "Probe"], {}, "could not open `missing.txt'"),
    ("an @ reference inside an @file", ["@outer.txt"], {"outer.txt": "-cp . @inner.txt\n", "inner.txt": "Probe\n"}, "Could not find or load main class @inner.txt"),
    ("an escaped @ as the main class", ["@@options.txt"], {}, "Could not find or load main class @options.txt"),
    ("a main class that does not exist", ["NoSuch"], {}, "Could not find or load main class NoSuch"),
    ("a main class with slashes and no main", ["java/lang/Object"], {}, "Main method not found in class java.lang.Object"),
    ("an option the JVM does not know", ["-t", "Probe"], {}, "Unrecognized option: -Xt"),
    ("an old form the JVM does not know", ["-oss2m", "Probe"], {}, "Unrecognized option: -Xoss2m"),
    ("an empty --add-opens=", ["--add-opens=", "Probe"], {}, "Unable to parse --add-opens"),
    ("a stack the JVM refuses", ["-Xss64k", "Probe"], {}, "The Java thread stack size specified is too small"),
]

rows, failures = [], []
for name, arguments, files, message in FAILURES:
    write(files)
    launcher = java(arguments)
    inside = embedded(arguments)
    said = (launcher.stdout + launcher.stderr, inside.stdout + inside.stderr)
    ok = (launcher.returncode != 0 and launcher.returncode == inside.returncode
          and message in said[0] and message in said[1] and not reports(launcher) and not reports(inside))
    rows.append((name, launcher.returncode, inside.returncode, "ok" if ok else "WRONG"))
    if not ok:
        failures.append((name, message, said))
show(rows, ("case", "java", "golem", ""))
for name, message, said in failures:
    print("\n--- %s, expected %r\n--- java\n%s--- golem\n%s" % (name, message, *said))
assert not failures, "%d of %d failures differ" % (len(failures), len(FAILURES))

## Options Golem refuses

Golem starts a main class in a JVM that serves a server, so it refuses with a clear error what only
the launcher can do or what Golem does not do: starting a jar, a module or a source file, printing a
version, the help or the settings, the reports on modules, a dry run, a splash screen and the choice
of another JVM or JRE. Each command line must make `launch-spec` exit with 1 and name the option.
A launch without a main class fails as well, where the launcher prints its usage.

In [ ]:
REFUSED = [
    ["-jar", "probe.jar"], ["-m", "java.base/java.lang.Object"], ["--module", "x/y"], ["--module=x/y"],
    ["--source", "17", "Probe.java"], ["--source=17", "Probe.java"], ["Probe.java"], ["Probe.JAVA"],
    ["-version"], ["--version"], ["-showversion", "Probe"], ["--show-version", "Probe"], ["-fullversion"], ["--full-version"],
    ["--dry-run", "Probe"], ["--list-modules"], ["--describe-module", "java.base"], ["--describe-module=java.base"], ["-d", "java.base"],
    ["--validate-modules"], ["--show-resolved-modules", "Probe"],
    ["-help"], ["-h"], ["-?"], ["--help"], ["-X"], ["--help-extra"],
    ["-XshowSettings", "Probe"], ["-XshowSettings:properties", "Probe"], ["-splash:image.png", "Probe"],
    ["-XXaltjvm=other", "Probe"], ["-J-XXaltjvm=other", "Probe"], ["-version:1.8", "Probe"], ["-jre-restrict-search", "Probe"], ["-jre-no-restrict-search", "Probe"],
]

# The launcher compares the suffix of a source file in any case, so Probe.JAVA is one too.
(SCRATCH / "Probe.JAVA").write_text((SCRATCH / "Probe.java").read_text(encoding="utf-8"), encoding="utf-8")
assert java(["Probe.JAVA"]).returncode == 0, "java does not run Probe.JAVA as a source file"
rows, failures = [], []
for arguments in REFUSED:
    golem, asked = spec(["-Dprobe.a=1", *arguments])
    ok = golem is None and asked.returncode == 1 and ("%s is not supported by Golem" % arguments[0]) in asked.stderr
    rows.append((" ".join(arguments), "refused" if ok else "WRONG"))
    if not ok:
        failures.append((arguments, asked.stdout + asked.stderr))
show(rows, ("command line", ""))
for arguments, said in failures:
    print("\n--- %s\n%s" % (arguments, said))
assert not failures, "%d of %d options are not refused" % (len(failures), len(REFUSED))
(SCRATCH / "Probe.JAVA").unlink()

none, asked = spec(["-Xmx64m"])
assert none is None and asked.returncode == 1 and "no main class" in asked.stderr, asked.stderr
assert java(["-Xmx64m"]).returncode == 1
print("no main class:", asked.stderr.strip().split(": ", 2)[-1])

## Where Golem differs on purpose

Each cell of this section shows one allowed difference with its reason.

- `--disable-@files` stops the expansion of @files. The launcher of Java 17 then passes the option
  on to the JVM, which refuses it as an unrecognized option, so the option cannot be used with java
  at all. Golem drops it after the expansion stops, so it does what its documentation says.
- `JDK_JAVA_OPTIONS` and `CLASSPATH` are variables of the launcher. Golem takes its launch from the
  files of the server alone, so the environment of a shell does not change what the server runs.
- The launcher warns on standard error that `-checksource`, `-cs` and `-noasyncgc` are no longer
  supported and that `-Xfuture` is deprecated. Golem drops the first three and translates the last
  as the launcher does, without the warnings, since a library of the host writes no lines of the
  launcher.

In [ ]:
write({"options.txt": "-Dprobe.file=1 -cp .\n"})
launcher = java(["--disable-@files", "-Dprobe.a=1", "Probe"])
assert launcher.returncode == 1 and "Unrecognized option: --disable-@files" in launcher.stderr, launcher.stderr
golem, _ = spec(["--disable-@files", "-Dprobe.a=1", "Probe", "x"])
assert golem == {"options": ["-Dprobe.a=1", "-Djava.class.path=."], "main_class": "Probe", "arguments": ["x"]}, golem
golem, _ = spec(["-Dprobe.a=1", "--disable-@files", "@options.txt"])
assert golem["main_class"] == "@options.txt", golem
inside = reports(embedded(["--disable-@files", "-Dprobe.a=1", "Probe"]))
assert inside and inside[0]["input"] == ["-Dprobe.a=1"] + HOOKS, inside
print("--disable-@files: the launcher fails with", launcher.stderr.strip().splitlines()[0])
print("--disable-@files: golem stops the expansion and drops the option")

variables = {"JDK_JAVA_OPTIONS": "-Dprobe.env=1"}
launcher = reports(java(["Probe"], variables=variables))
inside = reports(embedded(["Probe"], variables=variables))
assert launcher[0]["input"] == ["-Dprobe.env=1"] and inside[0]["input"] == HOOKS, (launcher, inside)
print("JDK_JAVA_OPTIONS: the launcher records %s, golem nothing" % launcher[0]["input"])

variables = {"CLASSPATH": ".:lib/a.jar"}
launcher = reports(java(["Probe"], variables=variables))
inside = reports(embedded(["Probe"], variables=variables))
assert launcher[0]["class_path"] == ".:lib/a.jar" and inside[0]["class_path"] == ".", (launcher, inside)
print("CLASSPATH: the launcher takes %s, golem ." % launcher[0]["class_path"])

old = ["-checksource", "-cs", "-noasyncgc", "-Xfuture", "Probe"]
launcher, inside = java(old), embedded(old)
warnings = [line for line in launcher.stderr.splitlines() if line.startswith("Warning: ")]
assert len(warnings) == 4 and not [line for line in inside.stderr.splitlines() if line.startswith("Warning: ")], (launcher.stderr, inside.stderr)
assert without_hooks(reports(inside)[0]) == reports(launcher)[0], "the old options change more than the warnings"
print("old options: the launcher warns %d times, golem not" % len(warnings))

## The Forge server

`launch-spec --server` reads the Forge server as its `run.sh` starts it:
`java @user_jvm_args.txt @libraries/net/minecraftforge/forge/<version>/unix_args.txt`. The cell
builds a directory of its own with a copy of `user_jvm_args.txt` and a link to the `libraries` of
the installed server, which nothing writes. The probe runs as a Java agent in front of the arguments
of Forge, prints its report before the main class runs and halts the JVM, under java and in the
embedded JVM, so Forge itself never starts. The launch must match the launcher's, the options of
the module system must take effect, and the boot layer must hold the same modules.

In [ ]:
SERVER = SCRATCH / "forge"
SERVER.mkdir()
shutil.copy(FORGE / "user_jvm_args.txt", SERVER / "user_jvm_args.txt")
(SERVER / "libraries").symlink_to(FORGE / "libraries", target_is_directory=True)
AGENT = "-javaagent:%s" % (SCRATCH / "probe.jar")
FORGE_ARGS = [AGENT, "@user_jvm_args.txt", "@" + UNIX_ARGS]

tokens = launch([DEVTOOL, "argfile", FORGE / "user_jvm_args.txt", FORGE / UNIX_ARGS], P.root)
assert tokens.returncode == 0, tokens.stderr
user_tokens, unix_tokens = [json.loads(line)["tokens"] for line in tokens.stdout.splitlines()]
assert user_tokens == [], "user_jvm_args.txt holds only comments, yet golem reads %s" % user_tokens

done = launch([DEVTOOL, "launch-spec", "--server", SERVER], P.root)
assert done.returncode == 0, done.stderr
golem = json.loads(done.stdout)
launcher = java(FORGE_ARGS, cwd=SERVER)
inside = embedded(FORGE_ARGS, cwd=SERVER)
report, = reports(launcher)
inside_report, = reports(inside)

assert report["probe"] == "agent" and report["input"][0] == AGENT, report["input"][:1]
assert recorded(golem["options"]) == report["input"][1:], "the options of Forge differ:\n%s\n%s" % (golem["options"], report["input"])
assert golem["main_class"] == "cpw.mods.bootstraplauncher.BootstrapLauncher", golem["main_class"]
assert " ".join([golem["main_class"], *golem["arguments"]]) == report["command"], report["command"]
assert last(golem["options"], "--module-path=") == report["module_path"], report["module_path"]
assert last(golem["options"], "-Djava.class.path=") == report["class_path"] == ".", report["class_path"]
assert report["checks"] and all(holds for _, holds in report["checks"]), report["checks"]
assert without_hooks(inside_report) == report, "the embedded report differs:\n%s\n%s" % (inside_report, report)
assert inside.returncode == launcher.returncode == 0, (inside.returncode, launcher.returncode)
assert "[golem] the JVM exits with status 0" in inside.stderr, inside.stderr
assert outcome(inside) is None, "jvm-run printed an outcome after Runtime.halt"
modules = [module["name"] for module in report["boot_modules"]]
print("tokens: user_jvm_args.txt %d, unix_args.txt %d" % (len(user_tokens), len(unix_tokens)))
print("options %d, arguments %d, main class %s" % (len(golem["options"]), len(golem["arguments"]), golem["main_class"]))
print("checks of --add-opens and --add-exports that hold: %d" % len(report["checks"]))
print("boot modules from the module path: %d, %s" % (len(modules), ", ".join(modules)))

## The host jar on the module path

`golem run` puts `golem-host.jar` on the module path of Forge with `add_module_path`, which appends
it to the last `--module-path`, the one the JVM takes. The launcher sees the same launch when the
`-p` line of `unix_args.txt` names the jar as well, so the cell writes such a copy for java and
gives the original to `launch-spec` and `jvm-run` with `--add-module-path`. The jar has no
`module-info.class` and names itself in its manifest, so it must become the automatic module
`com.skytekx.golem.host` of the boot layer, which exports its package, as `okf/bridge.md` relies on.

In [ ]:
HOST_JAR = SCRATCH / "golem-host.jar"
lines = (FORGE / UNIX_ARGS).read_text(encoding="utf-8").splitlines()
assert lines[0].startswith("-p "), "the first line of unix_args.txt is no -p: %s" % lines[0][:40]
(SERVER / "unix_args_host.txt").write_text("\n".join([lines[0] + ":" + str(HOST_JAR), *lines[1:]]) + "\n", encoding="utf-8")

done = launch([DEVTOOL, "launch-spec", "--server", SERVER, "--add-module-path", HOST_JAR], P.root)
assert done.returncode == 0, done.stderr
golem = json.loads(done.stdout)
assert last(golem["options"], "--module-path=").endswith(":" + str(HOST_JAR)), "the jar is not at the end of the module path"
assert sum(option.startswith("--module-path=") for option in golem["options"]) == 1, "a second --module-path appeared"

launcher = java([AGENT, "@user_jvm_args.txt", "@unix_args_host.txt"], cwd=SERVER)
inside = embedded(FORGE_ARGS, cwd=SERVER, jars=[HOST_JAR])
report, = reports(launcher)
inside_report, = reports(inside)
assert recorded(golem["options"]) == report["input"][1:], "the launch with the host jar differs"
assert without_hooks(inside_report) == report, "the embedded report differs:\n%s\n%s" % (inside_report, report)
host = [module for module in inside_report["boot_modules"] if module["name"] == "com.skytekx.golem.host"]
assert host == [{"automatic": True, "exported": ["com.skytekx.golem.host"], "name": "com.skytekx.golem.host"}], host
print("host module:", host[0])

## The embedded JVM

The same probe runs under java and in the embedded JVM, and the cell compares what each prints and
how each ends. When main returns while a thread that is no daemon still runs, `DestroyJavaVM` waits
for the thread and then runs the shutdown hooks, as the launcher does, and `jvm-run` prints its
outcome after both. `System.exit` ends the process with its status after the exit hook of the host
has logged it. An exception of main reaches the uncaught exception handler of the JVM, which prints
it as under the launcher, and `jvm-run` reports it in the outcome. A fatal error of the JVM, here an
`OutOfMemoryError` with `-XX:+CrashOnOutOfMemoryError`, passes the abort hook of the host. A crash
in native code, here a write to the address 0 through `Unsafe`, must give the fatal error report and
the `hs_err` file of the JVM as under the launcher, and no bare `SIGSEGV`, since the host puts
`SIGSEGV` and `SIGBUS` back to their default action before it creates the JVM.

In [ ]:
def lines(done):
    # The standard output of a run without the reports and the outcome.
    return [line for line in done.stdout.splitlines() if not line.startswith(("{", "["))]


def trace(done):
    # The lines of the uncaught exception handler on standard error.
    text = done.stderr.splitlines()
    start = [index for index, line in enumerate(text) if line.startswith("Exception in thread")]
    return [line for line in text[start[0]:] if line.startswith(("Exception in thread", "\tat "))] if start else []


rows = []
launcher, inside = java(["Probe", "thread", "hook"]), embedded(["Probe", "thread", "hook"])
assert launcher.returncode == inside.returncode == 0, (launcher.stderr, inside.stderr)
assert lines(launcher) == lines(inside) == ["probe: the thread ends", "probe: the shutdown hook runs"], (lines(launcher), lines(inside))
assert inside.stdout.splitlines()[-1].startswith('{"destroy_code"'), "the outcome came before the thread ended"
assert outcome(inside) == {"destroy_code": 0, "exception": None, "main": "returned"}, outcome(inside)
rows.append(("main returns, a thread and a hook run", launcher.returncode, inside.returncode, "ok"))

launcher, inside = java(["Probe", "hook", "exit=3"]), embedded(["Probe", "hook", "exit=3"])
assert launcher.returncode == inside.returncode == 3, (launcher.returncode, inside.returncode)
assert lines(launcher) == lines(inside) == ["probe: the shutdown hook runs"], (lines(launcher), lines(inside))
assert "[golem] the JVM exits with status 3" in inside.stderr and outcome(inside) is None, inside.stderr
rows.append(("System.exit(3)", launcher.returncode, inside.returncode, "ok"))

launcher, inside = java(["Probe", "throw"]), embedded(["Probe", "throw"])
assert launcher.returncode == 1 and inside.returncode == 0, (launcher.returncode, inside.returncode)
assert trace(launcher) and trace(launcher) == trace(inside), (trace(launcher), trace(inside))
assert outcome(inside) == {"destroy_code": 0, "exception": "java.lang.IllegalStateException: the probe throws", "main": "threw"}, outcome(inside)
rows.append(("main throws", launcher.returncode, "%d, %s" % (inside.returncode, outcome(inside)["main"]), "ok"))

fatal = ["-Xmx32m", "-XX:+CrashOnOutOfMemoryError", "-XX:-CreateCoredumpOnCrash", "-XX:ErrorFile=%s/hs_err_%%p.log" % SCRATCH, "Probe", "oom"]
launcher, inside = java(fatal), embedded(fatal)
assert launcher.returncode == inside.returncode != 0, (launcher.returncode, inside.returncode)
assert "[golem] the JVM aborts on a fatal error" in inside.stderr and outcome(inside) is None, inside.stderr
assert "[golem] the JVM aborts" not in launcher.stderr
rows.append(("a fatal error", launcher.returncode, inside.returncode, "ok"))

crash = ["-XX:-CreateCoredumpOnCrash", "-XX:ErrorFile=%s/hs_err_segv_%%p.log" % SCRATCH, "Probe", "segv"]
launcher = java(crash)
reports_java = sorted(SCRATCH.glob("hs_err_segv_*.log"))
inside = embedded(crash)
reports_golem = sorted(set(SCRATCH.glob("hs_err_segv_*.log")) - set(reports_java))
assert launcher.returncode == inside.returncode and inside.returncode not in (0, -11, 139), (launcher.returncode, inside.returncode)
assert len(reports_java) == 1 and len(reports_golem) == 1, "a crash left no hs_err file: %s %s" % (reports_java, reports_golem)
for done in (launcher, inside):
    assert "A fatal error has been detected by the Java Runtime Environment" in done.stdout + done.stderr, done.stdout[-2000:]
assert "[golem] the JVM aborts on a fatal error" in inside.stderr and outcome(inside) is None, inside.stderr
rows.append(("a crash in native code", launcher.returncode, inside.returncode, "ok"))
show(rows, ("case", "java", "golem", ""))

## The program golem

`golem source` names the version, the repository and the commit the binary is built from, which
`build.rs` takes from git, as section 13 of the AGPL asks. `golem run` finds the Java home in
`--java-home` or `JAVA_HOME`, the host jar in `--host-jar`, `GOLEM_HOST_JAR`, beside the program or
in the server directory, and refuses with a clear error what it cannot find, before any JVM starts.
A path may hold bytes that are not UTF-8, as a path of the system may, but a command and an argument
of main must be text, and one that is not is an error with the status 1 and no panic. The cell runs
only those paths, since a whole boot needs the bridge and the plugins, which other notebooks check.

In [ ]:
import tomllib

version = tomllib.loads((P.root / "Cargo.toml").read_text())["workspace"]["package"]["version"]
head = run(["git", "-C", P.root, "rev-parse", "HEAD"], check=True).stdout.strip()
source = launch([GOLEM, "source"], P.root)
assert source.returncode == 0, source.stderr
expected = ["version " + version, "repository https://github.com/skytekx/golem", "commit " + (os.environ.get("GOLEM_COMMIT") or head)]
assert source.stdout.splitlines() == expected, source.stdout
assert launch([GOLEM, "version"], P.root).stdout.strip() == "golem " + version

EMPTY = SCRATCH / "empty"
EMPTY.mkdir()
# A name that is not UTF-8, which a path of the system may hold.
NOT_UTF8 = SCRATCH / os.fsdecode(b"caf\xe9")
NOT_UTF8.mkdir()
assert not (GOLEM.parent / "golem-host.jar").exists(), "a golem-host.jar beside the program hides the error of a missing jar"
RUNS = [
    ("no Java home", ["run", "--server", SERVER], ["JAVA_HOME"], "no Java home: pass --java-home or set JAVA_HOME"),
    ("a Java home without libjvm", ["run", "--server", SERVER, "--java-home", EMPTY], [], "holds no"),
    ("no host jar", ["run", "--server", SERVER, "--java-home", JAVA_HOME], [], "no golem-host.jar: pass --host-jar, set GOLEM_HOST_JAR"),
    ("a host jar that does not exist", ["run", "--server", SERVER, "--java-home", JAVA_HOME, "--host-jar", EMPTY / "x.jar"], [], "the host jar"),
    ("no Forge server", ["run", "--server", EMPTY, "--java-home", JAVA_HOME, "--host-jar", HOST_JAR], [], "holds no Forge server"),
    ("a budget that is no number", ["run", "--budget-ms", "two"], [], "--budget-ms takes a whole number of milliseconds"),
    ("an unknown option", ["run", "--fast"], [], "run has no option --fast"),
    ("an unknown command", ["start"], [], "no command start"),
    ("a command that is not UTF-8", [os.fsdecode(b"caf\xe9")], [], "golem: no command caf\ufffd"),
    ("a server whose path is not UTF-8", ["run", "--server", NOT_UTF8, "--java-home", JAVA_HOME, "--host-jar", HOST_JAR], [],
     "caf\ufffd holds no Forge server"),
    ("an argument of main that is not UTF-8", ["run", "--", os.fsdecode(b"\xff")], [], "golem: the argument \ufffd after -- is not UTF-8"),
]
rows = []
for name, arguments, unset, message in RUNS:
    done = launch([GOLEM, *arguments], P.root, unset=unset)
    ok = done.returncode == 1 and message in done.stderr
    rows.append((name, done.returncode, "ok" if ok else "WRONG"))
    assert ok, "%s: %d\n%s" % (name, done.returncode, done.stderr)
show(rows, ("golem", "status", ""))

## Verdict

The cell clears the temporary directory and prints the counts. A run that reaches it passed every
case.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: golem-jvm reads %d argument files, %d launches, %d failing launches, %d refused command lines and the Forge server %s as the java launcher of Temurin %s does, and the embedded JVM runs the probe as java runs it, but for the 5 differences on purpose and the 4 warnings"
      % (len(ARGFILES), len(LAUNCHES), len(FAILURES), len(REFUSED), forge_version, java_version))